# Обучение модели СИЗ (каска + сигнальный жилет) — Google Colab

Модель YOLO11n дообучается на открытом датасете и выдаёт `best.pt` для программы Security Vision.

**Перед началом:** меню *Среда выполнения → Сменить тип среды выполнения → T4 GPU*.
Затем выполняйте ячейки сверху вниз (Shift+Enter). Всё занимает 30–45 минут, обучение ограничено по времени.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics roboflow openvino

## Шаг 1. Датасет

Нужен датасет в формате **YOLO** с классами «человек», «каска», «жилет» (лучше ещё «нет каски» / «нет жилета»).
Подходит Roboflow-датасет **Construction Site Safety** (классы Hardhat, NO-Hardhat, Safety Vest, NO-Safety Vest, Person и др.).

**Вариант А (проще всего): Roboflow.** Зарегистрируйтесь на roboflow.com, найдите датасет на universe.roboflow.com
(поиск: *construction site safety*), нажмите **Download Dataset → YOLOv8 → Show download code**
и перенесите `workspace`, `project`, `version` и ключ ниже.

**Вариант Б: zip-архив.** Скачайте датасет вручную (Roboflow или Kaggle), загрузите zip в Colab
(значок папки слева → «Загрузить») и укажите его имя.

In [ ]:
DATASET_SOURCE = "roboflow"   # "roboflow" или "zip"

if DATASET_SOURCE == "roboflow":
    ROBOFLOW_API_KEY = "ВСТАВЬТЕ_КЛЮЧ"
    WORKSPACE = "workspace-из-кода-на-сайте"
    PROJECT = "project-из-кода-на-сайте"
    VERSION = 1
    from roboflow import Roboflow
    Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location="/content/dataset")
else:
    ZIP_PATH = "/content/archive.zip"
    import zipfile
    zipfile.ZipFile(ZIP_PATH).extractall("/content/dataset")

In [ ]:
# Находим data.yaml, чинит пути (в выгрузках Roboflow они относительные и часто неверные) и проверяет классы
import glob, pathlib, re, yaml

yamls = glob.glob("/content/dataset/**/data.yaml", recursive=True)
assert yamls, "data.yaml не найден: датасет должен быть в формате YOLO"
DATA_YAML = yamls[0]
root = pathlib.Path(DATA_YAML).parent
cfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))

def first_existing(*names):
    for n in names:
        if (root / n / "images").exists():
            return str((root / n / "images").resolve())
    return None

paths = {"train": first_existing("train"), "val": first_existing("valid", "val"), "test": first_existing("test")}
assert paths["train"] and paths["val"], f"Не нашёл папки train/valid с images внутри {root}"
cfg.pop("path", None)
cfg.pop("test", None)
cfg.update({k: v for k, v in paths.items() if v})
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

names = cfg["names"]
names = list(names.values()) if isinstance(names, dict) else list(names)
print("Классы датасета:", names)

# проверка теми же правилами, по которым программа сопоставляет классы
norm = lambda s: re.sub(r"[\s_\-]+", " ", s.strip().lower())
wanted = {
    "person (человек)": ["person", "worker"],
    "helmet (каска)": ["hardhat", "hard hat", "helmet", "safety helmet"],
    "vest (жилет)": ["safety vest", "vest", "reflective vest"],
}
for role, aliases in wanted.items():
    found = any(norm(n) in [norm(a) for a in aliases] for n in names)
    print(("  OK   " if found else "  НЕТ  ") + role)
print("Разбиение test есть:" , "test" in cfg)

Если выше у какой-то роли стоит **НЕТ**, программа не сможет её определить. Выберите другой датасет
или (для опытных) добавьте названия классов в `model.class_map` в `config/default.json`.

## Шаг 2. Обучение

Обучение останавливается само через `MAX_MINUTES` минут (выбирается лучшая эпоха). Для 2–3 тысяч изображений хватает 30–35 минут.

In [ ]:
from ultralytics import YOLO

MAX_MINUTES = 35
IMGSZ = 640
BATCH = 16
EPOCHS = 100   # верхняя граница; фактически ограничено временем

model = YOLO("yolo11n.pt")   # самая лёгкая модель: потом она должна работать на процессоре ноутбука
model.train(data=DATA_YAML, imgsz=IMGSZ, epochs=EPOCHS, time=MAX_MINUTES / 60, batch=BATCH,
            patience=15, workers=2, project="/content/runs", name="ppe", exist_ok=True, seed=0, plots=True)

## Шаг 3. Метрики качества для отчёта

Оценка на отдельной тестовой выборке (если в датасете её нет, то на валидационной).

In [ ]:
import json

best = YOLO("/content/runs/ppe/weights/best.pt")
split = "test" if "test" in cfg else "val"
m = best.val(data=DATA_YAML, split=split, imgsz=IMGSZ, project="/content/runs", name="ppe_eval", exist_ok=True, plots=True)

per_class = {
    m.names[int(c)]: {"precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
                      "mAP50": float(m.box.ap50[i]), "mAP50-95": float(m.box.ap[i])}
    for i, c in enumerate(m.box.ap_class_index)
}
summary = {"split": split, "precision": float(m.box.mp), "recall": float(m.box.mr),
           "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map), "per_class": per_class}
with open("/content/metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Выборка: {split}")
print(f"{'класс':<18}{'precision':>10}{'recall':>9}{'mAP50':>8}{'mAP50-95':>10}")
print(f"{'ВСЕ':<18}{summary['precision']:>10.3f}{summary['recall']:>9.3f}{summary['mAP50']:>8.3f}{summary['mAP50-95']:>10.3f}")
for name, v in per_class.items():
    print(f"{name:<18}{v['precision']:>10.3f}{v['recall']:>9.3f}{v['mAP50']:>8.3f}{v['mAP50-95']:>10.3f}")

## Шаг 4. Упаковка результатов

Скачайте архив `ppe_result.zip`. В нём веса (`best.pt` и папка `best_openvino_model` — версия для OpenVINO, на процессорах Intel без видеокарты быстрее), метрики, графики и примеры работы модели для отчёта.

In [ ]:
import shutil

try:   # OpenVINO на части процессоров Intel быстрее, на других нет (проверяйте tools/benchmark.py); не критично, если не получится
    best.export(format="openvino", imgsz=IMGSZ)   # папка best_openvino_model рядом с best.pt
except Exception as exc:
    print("OpenVINO не создан (не критично):", exc)

out = pathlib.Path("/content/ppe_result")
shutil.rmtree(out, ignore_errors=True)
out.mkdir()

def put(src, prefix=""):
    src = pathlib.Path(src)
    if src.exists() and not src.name.startswith("train_batch"):
        shutil.copy(src, out / (prefix + src.name))

put("/content/runs/ppe/weights/best.pt")
if pathlib.Path("/content/runs/ppe/weights/best_openvino_model").exists():
    shutil.copytree("/content/runs/ppe/weights/best_openvino_model", out / "best_openvino_model")
for f in glob.glob("/content/runs/ppe/*.png") + glob.glob("/content/runs/ppe/*.jpg") + ["/content/runs/ppe/results.csv", "/content/runs/ppe/args.yaml"]:
    put(f, "train_")
for f in glob.glob("/content/runs/ppe_eval/*.png") + glob.glob("/content/runs/ppe_eval/*.jpg"):
    put(f, f"{split}_")
put("/content/metrics.json")

archive = shutil.make_archive("/content/ppe_result", "zip", out)
print("Готово:", archive, sorted(p.name for p in out.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## Что дальше

1. Распакуйте архив. Файл **`best.pt`** положите в папку `models/` проекта и переименуйте в **`ppe.pt`**.
2. Для ускорения: папку **`best_openvino_model`** тоже положите в `models/` и переименуйте в **`ppe_openvino_model`**
   (программа возьмёт её автоматически; см. README.md).
3. Графики и `metrics.json` пригодятся для раздела отчёта о качестве модели.
4. Запустите программу (`python main.py`).